In [2]:
import os, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Load data (local file or download)
url = "https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv"
df = pd.read_csv("creditcard.csv" if os.path.exists("creditcard.csv") else url)
df[["Amount", "Time"]] = RobustScaler().fit_transform(df[["Amount", "Time"]])
X, y = df.drop(columns="Class"), df["Class"]

# Split first, so the test set keeps the real imbalance
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

def undersample(X, y, ratio):
    fraud = X[y == 1]
    legit = X[y == 0].sample(len(fraud) * ratio, random_state=42)
    Xs = pd.concat([fraud, legit]).sample(frac=1, random_state=42)
    return Xs, y.loc[Xs.index]

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(probability=True),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
}

results = []
for ratio in [1, 2]:                      # 1:1 (material) vs 2:1 (exercise)
    Xu, yu = undersample(Xtr, ytr, ratio)
    for name, m in models.items():
        m.fit(Xu, yu)
        p, s = m.predict(Xte), m.predict_proba(Xte)[:, 1]
        results.append([f"{ratio}:1", name, recall_score(yte, p), precision_score(yte, p),
                        f1_score(yte, p), roc_auc_score(yte, s)])

print(pd.DataFrame(results, columns=["Ratio", "Model", "Recall", "Precision", "F1", "ROC-AUC"]).round(3))

c:\Users\ROWTECH\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ROWTECH\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\ROWTECH\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ROWTECH\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", l

  Ratio                Model  Recall  Precision     F1  ROC-AUC
0   1:1  Logistic Regression   0.918      0.038  0.074    0.976
1   1:1                  KNN   0.908      0.054  0.103    0.965
2   1:1                  SVC   0.888      0.071  0.131    0.979
3   1:1        Decision Tree   0.908      0.016  0.031    0.906
4   1:1        Random Forest   0.908      0.042  0.080    0.973
5   2:1  Logistic Regression   0.918      0.060  0.113    0.975
6   2:1                  KNN   0.908      0.096  0.173    0.957
7   2:1                  SVC   0.888      0.107  0.191    0.979
8   2:1        Decision Tree   0.857      0.021  0.041    0.894
9   2:1        Random Forest   0.898      0.079  0.144    0.974
